# IPL Exploratory Data Analysis (EDA)

**Mini Project | BCA | Python, Pandas & Matplotlib**

This project explores a learning dataset inspired by IPL match data. The dataset is **synthetically generated for this project** and should not be interpreted as official IPL records. It includes deliberate data-quality issues for demonstrating cleaning.


## 1. Project Objectives
- Explore match records and columns
- Check and correct data types
- Handle missing values and duplicates
- Identify and handle unusual values (outliers)
- Calculate KPIs and create visualizations
- Summarize insights and conclusions


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path('../data/ipl_matches_original.csv')
OUTPUT_PATH = Path('../outputs')
OUTPUT_PATH.mkdir(exist_ok=True)
df = pd.read_csv(DATA_PATH)


## 2. Dataset Overview

In [ ]:
display(df.head())
print('Rows and columns:', df.shape)
print('Column names:', df.columns.tolist())
df.info()
display(df.describe(include='all').T)


## 3. Data Types
Date is converted to datetime and season to integer.

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['season'] = pd.to_numeric(df['season'], errors='coerce').astype('Int64')
print(df.dtypes)


## 4. Missing Values
We inspect missing values and fill numeric columns with median, and text columns with the most frequent value.

In [ ]:
missing_before = df.isna().sum().sort_values(ascending=False)
display(missing_before[missing_before > 0])
clean = df.copy()
for col in clean.columns:
    if clean[col].isna().any():
        if pd.api.types.is_numeric_dtype(clean[col]):
            clean[col] = clean[col].fillna(clean[col].median())
        else:
            clean[col] = clean[col].fillna(clean[col].mode().iloc[0])
print('Missing values after filling:', int(clean.isna().sum().sum()))


## 5. Duplicate Records
We use match_id as the unique identifier and remove repeated records.

In [ ]:
duplicate_count = clean.duplicated(subset='match_id').sum()
print('Duplicate match IDs:', duplicate_count)
clean = clean.drop_duplicates(subset='match_id').reset_index(drop=True)
print('Shape after duplicate removal:', clean.shape)


## 6. Outlier Detection and Handling
The IQR method flags values outside 1.5 × IQR. We cap extreme values at the IQR bounds rather than deleting whole matches.

In [ ]:
numeric_cols = ['first_innings_runs','second_innings_runs','win_margin']
for col in numeric_cols:
    q1, q3 = clean[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = max(0, q1 - 1.5*iqr), q3 + 1.5*iqr
    outliers = ((clean[col] < low) | (clean[col] > high)).sum()
    print(f'{col}: outliers={outliers}, lower={low:.1f}, upper={high:.1f}')
    clean[col] = clean[col].clip(low, high).round().astype(int)


## 7. Clean Dataset Preview and Export

In [ ]:
display(clean.head())
print('Clean shape:', clean.shape)
clean.to_csv(OUTPUT_PATH / 'ipl_matches_cleaned.csv', index=False)


## 8. KPI Analysis
We calculate eight simple and meaningful KPIs.

In [ ]:
total_matches = clean['match_id'].nunique()
avg_first_innings = clean['first_innings_runs'].mean()
avg_second_innings = clean['second_innings_runs'].mean()
toss_win_match_win_rate = (clean['toss_winner'] == clean['winner']).mean() * 100
most_wins_team = clean['winner'].value_counts().idxmax()
most_wins_count = clean['winner'].value_counts().max()
chase_wins = (clean['win_type'] == 'wickets').sum()
defend_wins = (clean['win_type'] == 'runs').sum()
chasing_win_pct = chase_wins / total_matches * 100
avg_total_score = (clean['first_innings_runs'] + clean['second_innings_runs']).mean()
avg_win_margin = clean['win_margin'].mean()
print(f'1. Total matches analyzed: {total_matches}')
print(f'2. Average first-innings score: {avg_first_innings:.1f}')
print(f'3. Average second-innings score: {avg_second_innings:.1f}')
print(f'4. Toss winner also won: {toss_win_match_win_rate:.1f}%')
print(f'5. Most wins (team in dataset): {most_wins_team} ({most_wins_count})')
print(f'6. Wins while chasing: {chase_wins} ({chasing_win_pct:.1f}%)')
print(f'7. Wins while defending: {defend_wins} ({defend_wins/total_matches*100:.1f}%)')
print(f'8. Average combined runs per match: {avg_total_score:.1f}')
print(f'9. Average winning margin: {avg_win_margin:.1f}')


## 9. Visualizations
Each chart is saved to the outputs folder and displayed.

In [ ]:
# KPI 1: Matches by season
clean.groupby('season').size().plot(kind='bar', figsize=(10,5), title='Matches in Dataset by Season')
plt.xlabel('Season'); plt.ylabel('Match count'); plt.tight_layout(); plt.savefig(OUTPUT_PATH/'matches_by_season.png'); plt.show()


In [ ]:
# KPI 2: Most frequent winners
clean['winner'].value_counts().head(10).plot(kind='bar', figsize=(10,5), title='Match Wins by Team')
plt.xlabel('Team'); plt.ylabel('Wins'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.savefig(OUTPUT_PATH/'team_wins.png'); plt.show()


In [ ]:
# KPI 3: First vs second innings average scores
means = [clean['first_innings_runs'].mean(), clean['second_innings_runs'].mean()]
plt.figure(figsize=(6,4)); plt.bar(['First innings','Second innings'], means); plt.title('Average Innings Scores'); plt.ylabel('Runs'); plt.tight_layout(); plt.savefig(OUTPUT_PATH/'average_innings_scores.png'); plt.show()


In [ ]:
# KPI 4: Toss decision counts
clean['toss_decision'].value_counts().plot(kind='pie', autopct='%1.1f%%', figsize=(6,6), title='Toss Decision Distribution'); plt.ylabel(''); plt.tight_layout(); plt.savefig(OUTPUT_PATH/'toss_decisions.png'); plt.show()


In [ ]:
# KPI 5: Chasing vs defending wins
plt.figure(figsize=(6,4)); plt.bar(['Chasing (wickets)','Defending (runs)'], [chase_wins, defend_wins]); plt.title('Win Type Distribution'); plt.ylabel('Match wins'); plt.tight_layout(); plt.savefig(OUTPUT_PATH/'win_type.png'); plt.show()


In [ ]:
# KPI 6: Venue average first innings score (top 8 venues)
venue_avg = clean.groupby('venue')['first_innings_runs'].mean().sort_values(ascending=False).head(8)
venue_avg.plot(kind='bar', figsize=(10,5), title='Average First Innings Score by Venue (Top 8)'); plt.xlabel('Venue'); plt.ylabel('Average runs'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.savefig(OUTPUT_PATH/'venue_average_scores.png'); plt.show()


In [ ]:
# KPI 7: Win margin distribution
plt.figure(figsize=(8,4)); plt.hist(clean['win_margin'], bins=15); plt.title('Winning Margin Distribution'); plt.xlabel('Margin'); plt.ylabel('Matches'); plt.tight_layout(); plt.savefig(OUTPUT_PATH/'win_margin_distribution.png'); plt.show()


In [ ]:
# KPI 8: Player of the match awards (top 10)
clean['player_of_match'].value_counts().head(10).plot(kind='bar', figsize=(10,5), title='Most Player of the Match Awards'); plt.xlabel('Player'); plt.ylabel('Awards'); plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.savefig(OUTPUT_PATH/'player_awards.png'); plt.show()


## 10. Key Insights
The statements below are calculated from this synthetic dataset. They should be interpreted as patterns in the generated data only—not actual IPL findings.

In [ ]:
print(f'In this generated dataset, {toss_win_match_win_rate:.1f}% of matches are won by the toss winner.')
print(f'{chasing_win_pct:.1f}% of matches are won while chasing.')
print(f'{most_wins_team} has the highest win count in this sample ({most_wins_count}).')
print(f'Average first-innings score is {avg_first_innings:.1f} and second-innings score is {avg_second_innings:.1f}.')
print('Venue averages vary in this sample; use the bar chart to compare them.')


## 11. Conclusion
This mini project demonstrates the complete EDA workflow: data loading, exploration, type checking, missing-value treatment, duplicate removal, outlier handling, KPI calculation, visualizations, and insight writing. The project uses a synthetic dataset for practice, so it is not suitable for quoting official IPL statistics.
